# Datatón Tech for Good · InnoSoft 26–27
## Equipo: `Anotop`
**Integrantes:** Juan · Nico · Paco

---
### La pregunta
> **¿Cuánto de la brecha salarial bruta en Andalucía responde realmente a una diferencia de salario a igual trabajo y cuánto a la diferencia de horas trabajadas?**

### Datasets utilizados
| Código | Dataset | Fuente · Ámbito · Periodo | Propósito en el análisis |
|---|---|---|---|
| **DS-005** | Salario por sexo en Andalucía | IECA · MCVL 2024 (Andalucía) | Analizar salario bruto frente a salario equivalente a tiempo completo y percentiles. |
| **DS-004** | Salario por edad en Andalucía | IECA · MCVL 2024 (Andalucía) | Evolución de la diferencia salarial y jornada por tramos de edad. |
| **DS-012** | Indicadores de brecha salarial | Elaboración oficial (2022–2024) | Validación y control de consistencia oficial con las dos fórmulas de brecha. |
| **DS-013** | Microdatos sintéticos TIC | Simulación calibrada (3.000 obs.) | Descomposición del efecto horas y modelo econométrico ajustado por ocupación. |

### Fórmulas oficiales empleadas
- **Brecha sobre hombres (%):** `(Hombres − Mujeres) / Hombres × 100`
- **Brecha de hombres sobre mujeres (%):** `(Hombres / Mujeres − 1) × 100`
---

## 0 · Preparación del Entorno y Carga Universal de Datos
Configuración inicial, creación de directorios y función de lectura robusta (funciona tanto en local como en Google Colab sin necesidad de subir archivos manualmente).

In [1]:
import time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
try:
    from IPython.display import display
except Exception:
    display = print

INICIO = time.perf_counter()
URL_BASE = "https://innosoft-igualdad-sostenibilidad.vercel.app/data/csv/"
RESULTS_DIR = Path("resultados")
FIGS_DIR = Path("figuras")
RESULTS_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)

MAPEO_ARCHIVOS = {
    "DS-005": "DS-005_salario_sexo_andalucia_2024.csv",
    "DS-004": "DS-004_salario_edad_andalucia_2024.csv",
    "DS-012": "DS-012_brecha_salarial_indicadores.csv",
    "DS-013": "DS-013_microdatos_sinteticos_tic.csv"
}

def leer(nombre_dataset: str) -> pd.DataFrame:
    """Lee el dataset en local o lo descarga automáticamente de la web si se ejecuta en Colab."""
    archivo = MAPEO_ARCHIVOS.get(nombre_dataset, nombre_dataset if nombre_dataset.endswith(".csv") else f"{nombre_dataset}.csv")
    rutas_locales = [Path(archivo), Path("datos/csv") / archivo, Path("../datos/csv") / archivo]
    for ruta in rutas_locales:
        if ruta.exists():
            return pd.read_csv(ruta, sep=";", decimal=",", thousands=".", encoding="utf-8-sig", na_values=["."])
    # Fallback automático a la web oficial
    return pd.read_csv(URL_BASE + archivo, sep=";", decimal=",", thousands=".", encoding="utf-8-sig", na_values=["."])

print("✓ Entorno inicializado y cargador universal configurado.")

✓ Entorno inicializado y cargador universal configurado.


## 1 · Análisis Oficial Macro (Andalucía · IECA / MCVL 2024)
**Autor responsable:** Nico

Cálculo riguroso de las brechas salariales oficiales en Andalucía, comparación de salario bruto frente a equivalente a tiempo completo, contraste media vs. mediana y validación matemática automática con el catálogo de indicadores `DS-012`.

In [2]:
def brecha_oficial(hombres: float, mujeres: float) -> dict:
    """Calcula la brecha según las dos definiciones oficiales exigidas en las bases."""
    return {
        "brecha_sobre_hombres_pct": round(((hombres - mujeres) / hombres) * 100, 2),
        "brecha_hombres_sobre_mujeres_pct": round(((hombres - mujeres) / mujeres) * 100, 2)
    }

def valor_positivo(fila: pd.Series, columna: str) -> float:
    """Conversión segura y positiva a valor numérico flotante."""
    return abs(float(fila[columna]))

def fila_brecha(nombre: str, hombres: float, mujeres: float, periodo: int, fuente: str) -> dict:
    brechas = brecha_oficial(hombres, mujeres)
    return {
        "indicador": nombre,
        "hombres_eur": hombres,
        "mujeres_eur": mujeres,
        "diferencia_eur": round(hombres - mujeres, 2),
        "brecha_sobre_hombres_pct": brechas["brecha_sobre_hombres_pct"],
        "brecha_hombres_sobre_mujeres_pct": brechas["brecha_hombres_sobre_mujeres_pct"],
        "periodo": periodo,
        "territorio": "Andalucía",
        "fuente": fuente
    }

def comprobar_con_ds012(resultados: pd.DataFrame, referencia: pd.DataFrame) -> None:
    """Valida automáticamente que nuestros cálculos coinciden con la tabla oficial de indicadores."""
    medidas = {
        "Salario bruto anual (€)": "Salario bruto anual medio",
        "Salario equivalente anual a tiempo completo (€)": "Salario equivalente a tiempo completo",
        "Mediana del salario bruto anual (€)": "Mediana bruta anual",
        "Mediana del salario equivalente (€)": "Mediana equivalente"
    }
    for medida, nombre in medidas.items():
        filas = referencia.loc[
            (referencia["territorio"] == "Andalucía") &
            (referencia["periodo"] == 2024) &
            (referencia["medida"] == medida)
        ]
        if len(filas) != 1:
            raise ValueError(f"DS-012 debería contener una fila para {medida!r}; se encontraron {len(filas)}.")
        
        calculado = resultados.loc[resultados["indicador"] == nombre]
        if len(calculado) != 1:
            raise ValueError(f"No se encontró un único resultado para {nombre!r}.")
        
        fila_ref = filas.iloc[0]
        fila_calc = calculado.iloc[0]
        columnas = (
            ("brecha_sobre_hombres", "brecha_sobre_hombres_pct"),
            ("brecha_hombres_sobre_mujeres", "brecha_hombres_sobre_mujeres_pct")
        )
        for columna_ref, columna_calc in columnas:
            valor_ref = pd.to_numeric(fila_ref[columna_ref], errors="coerce")
            valor_calc = float(fila_calc[columna_calc])
            if pd.isna(valor_ref) or abs(float(valor_ref) - valor_calc) > 0.01:
                raise ValueError(f"El resultado de {nombre!r} no coincide con DS-012: ref={valor_ref}, calc={valor_calc}")

# Carga de datasets oficiales
salario_sexo = leer("DS-005")
salario_edad = leer("DS-004")
indicadores = leer("DS-012")

hombres_df = salario_sexo.loc[salario_sexo["sexo"] == "Hombres"]
mujeres_df = salario_sexo.loc[salario_sexo["sexo"] == "Mujeres"]
hombres = hombres_df.iloc[0]
mujeres = mujeres_df.iloc[0]

definiciones = [
    ("Salario bruto anual medio", "salario_bruto_anual", "IECA · MCVL 2024 · DS-005"),
    ("Salario equivalente a tiempo completo", "salario_eq_tiempo_completo", "IECA · MCVL 2024 · DS-005"),
    ("Mediana bruta anual", "bruto_p50", "IECA · MCVL 2024 · DS-005"),
    ("Mediana equivalente", "eq_p50", "IECA · MCVL 2024 · DS-005"),
]

brechas_oficiales = pd.DataFrame([
    fila_brecha(nombre, valor_positivo(hombres, columna), valor_positivo(mujeres, columna), int(hombres["periodo"]), fuente)
    for nombre, columna, fuente in definiciones
])

# Ejecutar verificación de consistencia con DS-012
comprobar_con_ds012(brechas_oficiales, indicadores)
print("✓ Validación con DS-012 completada con éxito: Consistencia 100% verificada.")
display(brechas_oficiales[["indicador", "hombres_eur", "mujeres_eur", "diferencia_eur", "brecha_sobre_hombres_pct", "brecha_hombres_sobre_mujeres_pct"]])

✓ Validación con DS-012 completada con éxito: Consistencia 100% verificada.


,indicador,hombres_eur,mujeres_eur,diferencia_eur,brecha_sobre_hombres_pct,brecha_hombres_sobre_mujeres_pct
0,Salario bruto anual medio,21103.0,17093.0,4010.0,19.00,23.46
1,Salario equivalente a tiempo completo,27182.0,24945.0,2237.0,8.23,8.97
2,Mediana bruta anual,18148.0,14038.0,4110.0,22.65,29.28
3,Mediana equivalente,22189.0,20481.0,1708.0,7.70,8.34


## 2 · Análisis Microdatos TIC y Descomposición del Efecto Horas (DS-013)
**Autor responsable:** Paco

Exploración de la distribución salarial individual en el sector TIC andaluz (3.000 observaciones sintéticas calibradas) y descomposición matemática formal entre el efecto del salario/hora y las horas semanales trabajadas.

In [3]:
df = leer("DS-013")

# Resumen por sexo
resumen_tic = df.groupby("sexo")[
    ["salario_bruto_anual", "salario_hora", "horas_semana"]
].agg(["mean", "median", "count"])
print("--- Resumen estadístico de Microdatos TIC ---")
display(resumen_tic.round(2))

# Cálculo de brechas en el sector TIC
medias = df.groupby("sexo")[["salario_bruto_anual", "salario_hora", "horas_semana"]].mean()
h = medias.loc["Hombre"]
m = medias.loc["Mujer"]

brechas_tic = pd.Series({
    "Brecha salarial anual": (h["salario_bruto_anual"] - m["salario_bruto_anual"]) / h["salario_bruto_anual"],
    "Brecha por hora": (h["salario_hora"] - m["salario_hora"]) / h["salario_hora"],
    "Brecha de horas": (h["horas_semana"] - m["horas_semana"]) / h["horas_semana"],
})
print("\n--- Brechas brutas en Sector TIC (%) ---")
display((brechas_tic * 100).round(2))

# Descomposición matemática del efecto salario por hora vs efecto horas trabajadas
salario_h_horas_m = h["salario_hora"] * m["horas_semana"]
salario_h_horas_h = h["salario_hora"] * h["horas_semana"]

efecto_horas = (h["salario_hora"] * (h["horas_semana"] - m["horas_semana"])) / salario_h_horas_h
efecto_salario_hora = ((h["salario_hora"] - m["salario_hora"]) * m["horas_semana"]) / salario_h_horas_h

peso_hora = efecto_salario_hora / (efecto_salario_hora + efecto_horas)
peso_horas = efecto_horas / (efecto_salario_hora + efecto_horas)

descomposicion = pd.Series({
    "Peso del salario por hora (%)": peso_hora * 100,
    "Peso de las horas trabajadas (%)": peso_horas * 100,
    "Total descompuesto (%)": (peso_hora + peso_horas) * 100
})
print("\n--- Descomposición de la diferencia anual TIC ---")
display(descomposicion.round(2))

# Comparación dentro de cada ocupación técnica
por_ocupacion = df.pivot_table(index="ocupacion", columns="sexo", values="salario_hora", aggfunc="mean")
por_ocupacion["brecha_hora_%"] = ((por_ocupacion["Hombre"] - por_ocupacion["Mujer"]) / por_ocupacion["Hombre"] * 100)
print("\n--- Brecha por hora según Ocupación Técnica (%)")
display(por_ocupacion.round(2))

--- Resumen estadístico de Microdatos TIC ---


salario_bruto_anual                salario_hora               \
                      mean   median count         mean median count   
sexo                                                                  
Hombre            31003.89  28805.0  2130        15.11  14.08  2130   
Mujer             24093.91  22130.5   870        12.13  11.34   870   

       horas_semana               
               mean median count  
sexo                              
Hombre        39.43   40.0  2130  
Mujer         38.26   40.0   870


--- Brechas brutas en Sector TIC (%) ---


Brecha salarial anual    22.29
Brecha por hora          19.71
Brecha de horas           2.96
dtype: float64


--- Descomposición de la diferencia anual TIC ---


Peso del salario por hora (%)        86.6
Peso de las horas trabajadas (%)     13.4
Total descompuesto (%)              100.0
dtype: float64


--- Brecha por hora según Ocupación Técnica (%)


sexo,Hombre,Mujer,brecha_hora_%
ocupacion,,,
Cloud y sistemas,15.99,14.45,9.67
Datos e IA,16.88,13.23,21.62
Desarrollo de software,14.60,13.21,9.54
Gestión y dirección,18.40,16.70,9.21
Otras tareas técnicas,11.74,10.06,14.24
Producto y UX,13.81,12.29,11.04
Soporte y sistemas,11.01,10.10,8.29


## 3 · Modelo Econométrico Ajustado (Ecuación de Mincer)
**Autor responsable:** Paco

Estimación de una regresión log-lineal semiparamétrica sobre el logaritmo del salario por hora para aislar el efecto independiente del género una vez controlados los factores objetivos de capital humano (experiencia, antigüedad), dedicación horaria y segregación ocupacional.

In [4]:
numericas = ["edad", "anios_experiencia", "antiguedad_empresa", "horas_semana"]
categoricas = ["tipo_contrato", "jornada", "ocupacion", "tamano_empresa", "provincia", "teletrabajo"]

X = pd.concat([
    (df["sexo"] == "Mujer").astype(int).rename("mujer"),
    df[numericas],
    pd.get_dummies(df[categoricas], drop_first=True, dtype=int)
], axis=1)

y = np.log(df["salario_hora"])
modelo = LinearRegression().fit(X, y)

coef_mujer = modelo.coef_[X.columns.get_loc("mujer")]
brecha_ajustada = (1 - np.exp(coef_mujer)) * 100
r2 = modelo.score(X, y)

print("--- RESULTADOS DEL MODELO ECONOMÉTRICO AJUSTADO ---")
print(f"R² del modelo (capacidad explicativa): {r2:.3f}")
print(f"Brecha salarial por hora bruta (sin controles): {((h['salario_hora'] - m['salario_hora']) / h['salario_hora'] * 100):.2f} %")
print(f"Brecha salarial ajustada (tras controlar experiencia, horas y rol): {brecha_ajustada:.2f} %")
print(f"Reducción de la brecha no explicada: {(1 - brecha_ajustada / ((h['salario_hora'] - m['salario_hora']) / h['salario_hora'] * 100)) * 100:.1f} %")

--- RESULTADOS DEL MODELO ECONOMÉTRICO AJUSTADO ---
R² del modelo (capacidad explicativa): 0.444
Brecha salarial por hora bruta (sin controles): 19.71 %
Brecha salarial ajustada (tras controlar experiencia, horas y rol): 10.55 %
Reducción de la brecha no explicada: 46.5 %


## 4 · Visualizaciones del Análisis
**Autor responsable:** Juan

Generación y exportación automática a `figuras/` de los 3 gráficos de conclusiones para las diapositivas de entrega.

In [5]:
# Estilo visual común (sobrio, tipografía de sistema, sin marco)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial", "DejaVu Sans"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#86868b", "axes.labelcolor": "#1d1d1f",
    "xtick.color": "#6e6e73", "ytick.color": "#6e6e73", "text.color": "#1d1d1f",
    "figure.facecolor": "white", "savefig.facecolor": "white",
})

# 1. Efecto jornada en Andalucía (Oficial MCVL 2024)
fig, ax = plt.subplots(figsize=(7.5, 4.2))
indicadores_n = ["Salario Bruto Anual Medio", "Salario Eq. Tiempo Completo"]
h_b = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario bruto anual medio", "hombres_eur"].values[0]
m_b = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario bruto anual medio", "mujeres_eur"].values[0]
h_eq = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario equivalente a tiempo completo", "hombres_eur"].values[0]
m_eq = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario equivalente a tiempo completo", "mujeres_eur"].values[0]

x = np.arange(len(indicadores_n))
width = 0.35
ax.bar(x - width/2, [h_b, h_eq], width, label="Hombres", color="#0071e3")
ax.bar(x + width/2, [m_b, m_eq], width, label="Mujeres", color="#bf5af2")
ax.set_ylabel("Euros anuales (€)")
ax.set_xticks(x)
ax.set_xticklabels(indicadores_n, fontweight="bold")
ax.set_ylim(0, 32000)
ax.legend(frameon=False)
ax.annotate("Brecha: 19,0%\n(H: 21.103€ vs M: 17.093€)", xy=(0, 21500), ha="center", fontsize=9, fontweight="bold", color="#333333")
ax.annotate("Brecha: 8,2%\n(H: 27.182€ vs M: 24.945€)\n[-56,8% de reducción]", xy=(1, 27500), ha="center", fontsize=9, fontweight="bold", color="#0071e3")
fig.suptitle("El efecto jornada: la brecha cae del 19,0% al 8,2% a tiempo completo", fontsize=11, fontweight="bold", x=0.02, ha="left")
fig.text(0.02, 0.01, "Fuente: IECA · MCVL 2024 (DS-005). Asalariados en Andalucía.", fontsize=8, color="gray")
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
fig.savefig(FIGS_DIR / "grafico1_efecto_jornada_andalucia.png", dpi=200)
plt.close()

# 2. Brecha por ocupacion TIC (Microdatos DS-013)
fmt = lambda v, d=1: f"{v:.{d}f}".replace(".", ",")
n_mujeres = df[df["sexo"] == "Mujer"].groupby("ocupacion").size()
tabla_ocup = por_ocupacion.sort_values("brecha_hora_%")
fig, ax = plt.subplots(figsize=(8, 4.2))
max_rol = tabla_ocup["brecha_hora_%"].idxmax()
colors = ["#ff375f" if rol == max_rol else "#d2d2d7" for rol in tabla_ocup.index]
etiquetas = [f"{rol} · {n_mujeres.get(rol, 0)} mujeres" for rol in tabla_ocup.index]
bars = ax.barh(etiquetas, tabla_ocup["brecha_hora_%"], color=colors)
ax.set_xlabel("Brecha salarial por hora (%) [(H-M)/H]")
ax.set_xlim(0, 30)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, f"{fmt(w)}%", va="center", fontweight="bold", fontsize=9)
fig.suptitle(f"Brecha salarial por hora según rol TIC: máxima en {max_rol} ({fmt(tabla_ocup.loc[max_rol, 'brecha_hora_%'])}%)", fontsize=11, fontweight="bold", x=0.02, ha="left")
fig.text(0.02, 0.01, "Fuente: microdatos SINTÉTICOS calibrados TIC (DS-013), 3.000 obs. Con menos de 50 mujeres en un rol, la cifra es poco fiable.", fontsize=7.5, color="gray")
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
fig.savefig(FIGS_DIR / "grafico2_brecha_por_ocupacion_tic.png", dpi=200)
plt.close()

# 3. Brecha antes y después de igualar condiciones (cada comparación dentro de su propio dataset)
brecha_and_bruta = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario bruto anual medio", "brecha_sobre_hombres_pct"].values[0]
brecha_and_eq = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Salario equivalente a tiempo completo", "brecha_sobre_hombres_pct"].values[0]
brecha_tic_bruta = float((h["salario_hora"] - m["salario_hora"]) / h["salario_hora"] * 100)
brecha_tic_aj = float(brecha_ajustada)
red_and = (1 - brecha_and_eq / brecha_and_bruta) * 100
red_tic = (1 - brecha_tic_aj / brecha_tic_bruta) * 100

grupos = ["Andalucía, todos los sectores\n(salario anual · DS-005, oficial)", "Sector TIC\n(salario por hora · DS-013, sintético)"]
antes = [brecha_and_bruta, brecha_tic_bruta]
despues = [brecha_and_eq, brecha_tic_aj]
reducciones = [red_and, red_tic]
x = np.arange(len(grupos))
width = 0.36
fig, ax = plt.subplots(figsize=(7.5, 4.2))
b1 = ax.bar(x - width/2, antes, width, color="#0071e3", label="Brecha sin ajustar")
b2 = ax.bar(x + width/2, despues, width, color="#a8d1ff", label="Brecha a igualdad de condiciones")
for bar in list(b1) + list(b2):
    v = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.4, f"{fmt(v)}%", ha="center", fontweight="bold", fontsize=9)
for i, (r, ctrl) in enumerate(zip(reducciones, ["igualando jornada", f"con modelo (R²={fmt(r2, 2)})"])):
    ax.text(x[i], antes[i] + 2.6, f"−{fmt(r, 0)}% {ctrl}", ha="center", fontsize=8.5, color="#333333")
ax.set_xticks(x)
ax.set_xticklabels(grupos, fontsize=9)
ax.set_ylabel("Brecha salarial (%) [(H-M)/H]")
ax.set_ylim(0, 30)
ax.legend(frameon=False, fontsize=8.5, loc="upper center", ncol=2)
fig.suptitle(f"Igualando condiciones, la brecha se reduce un {fmt(red_and, 0)}% (jornada) y un {fmt(red_tic, 0)}% (modelo TIC)", fontsize=11, fontweight="bold", x=0.02, ha="left")
fig.text(0.02, 0.01, "Fuentes: IECA · MCVL 2024 (DS-005) y regresión log-lineal sobre microdatos sintéticos TIC (DS-013). No comparar barras entre grupos.", fontsize=7.5, color="gray")
fig.tight_layout(rect=(0, 0.04, 1, 0.95))
fig.savefig(FIGS_DIR / "grafico3_descomposicion_regresion.png", dpi=200)
plt.close()

print("✓ Los 3 gráficos han sido exportados correctamente a figuras/.")

✓ Los 3 gráficos han sido exportados correctamente a figuras/.


## 5 · Exportación del CSV de Resultados Oficiales
**Autor responsable:** Juan

Guardado automático del archivo entregable consolidado `resultados/Anotop_resultados.csv` con todos los indicadores, unidades y notas metodológicas.

In [6]:
h_p50 = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Mediana bruta anual", "hombres_eur"].values[0]
m_p50 = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Mediana bruta anual", "mujeres_eur"].values[0]
h_eq50 = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Mediana equivalente", "hombres_eur"].values[0]
m_eq50 = brechas_oficiales.loc[brechas_oficiales["indicador"] == "Mediana equivalente", "mujeres_eur"].values[0]

res_final = pd.DataFrame([
    {"ambito": "Andalucía", "fuente": "IECA · MCVL 2024 (DS-005)", "indicador": "Salario bruto anual medio", "hombres_eur": h_b, "mujeres_eur": m_b, "diferencia_eur": round(h_b - m_b, 2), "brecha_sobre_hombres_pct": 19.0, "brecha_hombres_sobre_mujeres_pct": 23.46, "efecto_aislado": "Brecha no ajustada (incluye jornada parcial)"},
    {"ambito": "Andalucía", "fuente": "IECA · MCVL 2024 (DS-005)", "indicador": "Salario eq. tiempo completo", "hombres_eur": h_eq, "mujeres_eur": m_eq, "diferencia_eur": round(h_eq - m_eq, 2), "brecha_sobre_hombres_pct": 8.23, "brecha_hombres_sobre_mujeres_pct": 8.97, "efecto_aislado": "Elimina jornada (-56.8% de reducción)"},
    {"ambito": "Andalucía", "fuente": "IECA · MCVL 2024 (DS-005)", "indicador": "Mediana bruta anual (P50)", "hombres_eur": h_p50, "mujeres_eur": m_p50, "diferencia_eur": round(h_p50 - m_p50, 2), "brecha_sobre_hombres_pct": 22.65, "brecha_hombres_sobre_mujeres_pct": 29.28, "efecto_aislado": "Persona típica"},
    {"ambito": "Andalucía", "fuente": "IECA · MCVL 2024 (DS-005)", "indicador": "Mediana equivalente (P50)", "hombres_eur": h_eq50, "mujeres_eur": m_eq50, "diferencia_eur": round(h_eq50 - m_eq50, 2), "brecha_sobre_hombres_pct": 7.7, "brecha_hombres_sobre_mujeres_pct": 8.34, "efecto_aislado": "Mediana a tiempo completo"},
    {"ambito": "Sector TIC", "fuente": "Microdatos calibrados (DS-013)", "indicador": "Brecha Salario Hora Bruto", "hombres_eur": round(float(h['salario_hora']), 2), "mujeres_eur": round(float(m['salario_hora']), 2), "diferencia_eur": round(float(h['salario_hora'] - m['salario_hora']), 2), "brecha_sobre_hombres_pct": 19.71, "brecha_hombres_sobre_mujeres_pct": 24.55, "efecto_aislado": "Brecha horaria sin ajustar"},
    {"ambito": "Sector TIC", "fuente": "Modelo Regresión (DS-013)", "indicador": "Brecha Ajustada por Modelo", "hombres_eur": None, "mujeres_eur": None, "diferencia_eur": None, "brecha_sobre_hombres_pct": round(float(brecha_ajustada), 2), "brecha_hombres_sobre_mujeres_pct": round((1 / (1 - float(brecha_ajustada)/100) - 1) * 100, 2), "efecto_aislado": "Ajustada por edad, horas, exp, antiguedad y rol (R2=0.444)"}
])

res_final.to_csv(RESULTS_DIR / "Anotop_resultados.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
print("✓ Anotop_resultados.csv guardado con éxito en resultados/.")
print(f"Tiempo de ejecución total: {time.perf_counter() - INICIO:.2f} s")

✓ Anotop_resultados.csv guardado con éxito en resultados/.
Tiempo de ejecución total: 0.25 s


## 6 · Conclusiones, Límites y Propuesta de Acción

### Conclusiones Principales
1. **El factor jornada explica más de la mitad de la brecha salarial:** En Andalucía (IECA MCVL 2024), la brecha bruta anual del **19,0 %** se reduce al **8,2 %** al medir el salario equivalente a tiempo completo. El 56,8 % de la diferencia responde a diferencias en horas trabajadas (mayor prevalencia de jornada parcial en mujeres).
2. **En el sector TIC, el rol técnico y la experiencia reducen la brecha ajustada al 10,55 %:** En los microdatos sintéticos calibrados (DS-013), la brecha horaria bruta es del 19,71 %, pero tras controlar econométricamente por experiencia, horas semanales y especialidad técnica (modelo log-lineal con R²=0,444), la brecha no explicada cae a un 10,55 %.
3. **El efecto mediana:** Al eliminar los valores atípicos de salarios muy elevados y medir la persona típica a tiempo completo, la brecha cae al 7,7 %.

### Qué NO se puede concluir (Límites metodológicos)
- **No se puede concluir que el 8,2 % restante sea discriminación salarial directa pura:** El salario equivalente a tiempo completo corrige la duración de la jornada, pero no corrige la segregación vertical (acceso a puestos directivos) ni horizontal (distribución entre subsectores).
- **Limitación de los microdatos:** DS-013 son datos sintéticos calibrados, no registros reales de personas individuales.
- **Brecha de datos oficial:** No existe una tabla oficial en Andalucía que cruce simultáneamente sector TIC × sexo × edad × tipo de contrato × horas efectivas.

### Propuesta de Acción Concreta
> **Acción para empresas y administraciones TIC:** Exigir en las auditorías retributivas y convenios del sector tecnológico andaluz la publicación obligatoria desagregada del **salario por hora efectiva y por tramos de jornada**, en lugar de presentar únicamente cifras agregadas anuales brutas, permitiendo identificar si la desigualdad se debe a barreras de conciliación (jornadas reducidas) o a políticas de promoción interna.